# RLBench: The Robot Learning Benchmark & Learning Environment

**A Self-Contained Tutorial Notebook**

---

**Paper:** Stephen James, Zicong Ma, David Rovick Arrojo, Andrew J. Davison (2019)  
**From:** Dyson Robotics Lab, Imperial College London  
**arXiv:** 1909.12271  
**Website:** https://sites.google.com/view/rlbench

---

## What You Will Learn

This notebook walks you through the key ideas behind RLBench from the ground up:

1. **Motivation** — Why a unified robot learning benchmark is needed
2. **Benchmark Properties** — The 6 key design principles
3. **Architecture Overview** — Scene, Tasks, Variations, Episodes, Environment API
4. **Observation & Action Spaces** — What the robot sees and how it acts
5. **Demonstrations** — How infinite demos are generated via motion planning
6. **Task Builder** — How to create new tasks using PyRep
7. **The Few-Shot Challenge** — The central benchmark challenge of the paper
8. **Other Research Applications** — RL, Imitation Learning, Sim-to-Real, SLAM
9. **Hands-On Simulations** — Visualising core concepts with Python code
10. **Exercises** — Guided challenges to test understanding

---

> **Prerequisites:** Basic Python, NumPy, and Matplotlib familiarity. No robotics background needed!

## Section 0 — Environment Setup

In [ ]:
# Install required packages
# !pip install numpy matplotlib scipy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Circle
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
print("All packages loaded successfully!")

---
## Section 1 — Motivation: Why Do We Need RLBench?

### The Problem with Existing Benchmarks

Before RLBench, robot learning researchers faced a big practical problem. Each research group:
- Created their **own** custom tasks to test their methods
- Could not compare results with other groups fairly
- Could accidentally make tasks **easier** to suit their method
- Had no standard measure of what counts as "good" performance

Existing benchmarks like **OpenAI Gym** and **DeepMind Control Suite** existed, but they focused on toy tasks (controlling a pendulum, balancing a cart) — very different from real-world tasks a household robot would need to perform.

### The RLBench Solution

RLBench provides:
- **100 completely unique, hand-designed tasks** (reaching, door opening, stacking, cooking, etc.)
- A **single robot** (Franka Panda arm) for all tasks — fair comparison
- **Infinite demonstrations** generated automatically via motion planning
- Support for **many research areas** in one place: RL, imitation learning, few-shot learning, SLAM

| Benchmark | Tasks | Domain | Demonstrations | Few-Shot Support |
|---|---|---|---|---|
| OpenAI Gym | ~20 | Toy (pendulum, cartpole) | None | No |
| DM Control Suite | ~30 | Toy (physics) | None | No |
| RoboTurk | 3 | Manipulation | Crowdsourced | No |
| Meta-World | 50 | Manipulation | Limited | Partial |
| **RLBench** | **100** | **Real-world manipulation** | **Infinite (auto-generated)** | **Yes (v1.0 challenge)** |

In [ ]:
# ─── Figure 1: Motivation — Benchmark Comparison ─────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Figure 1 — Why RLBench? Comparing Robot Learning Benchmarks',
             fontsize=13, fontweight='bold', y=1.01)

# ──── Panel A: Number of tasks per benchmark ────
ax = axes[0]
benchmarks = ['OpenAI\nGym', 'DM Control\nSuite', 'RoboTurk', 'Simitate', 'Meta-World', 'RLBench']
n_tasks    = [6, 28, 3, 12, 50, 100]
colors_bm  = ['#90A4AE', '#90A4AE', '#90A4AE', '#90A4AE', '#64B5F6', '#EF5350']
bars = ax.bar(benchmarks, n_tasks, color=colors_bm, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, n_tasks):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1.5, str(val),
            ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Number of Tasks', fontsize=10)
ax.set_title('A) Task Count\nAcross Benchmarks', fontsize=11)
ax.set_ylim(0, 120)
ax.tick_params(axis='x', labelsize=8)
ax.grid(axis='y', alpha=0.3)
ax.annotate('10× more tasks\nthan RoboTurk!',
            xy=(5, 100), xytext=(3.5, 110),
            fontsize=8, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred'))

# ──── Panel B: Research areas supported ────
ax2 = axes[1]
research_areas = ['Reinforcement\nLearning', 'Imitation\nLearning', 'Few-Shot\nLearning',
                  'Multi-Task\nLearning', 'Sim-to-Real', 'SLAM']
supports = {
    'OpenAI Gym':  [1, 0, 0, 0, 0, 0],
    'Meta-World':  [1, 1, 1, 1, 0, 0],
    'RLBench':     [1, 1, 1, 1, 1, 1],
}
x = np.arange(len(research_areas))
width = 0.28
colors3 = ['#90A4AE', '#64B5F6', '#EF5350']
for i, (name, vals) in enumerate(supports.items()):
    ax2.bar(x + i*width, vals, width, label=name, color=colors3[i],
            edgecolor='black', linewidth=0.7)
ax2.set_xticks(x + width)
ax2.set_xticklabels(research_areas, fontsize=8)
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Not Supported', 'Supported'], fontsize=9)
ax2.set_title('B) Research Areas\nSupported', fontsize=11)
ax2.legend(fontsize=8)
ax2.set_ylim(0, 1.5)
ax2.grid(axis='y', alpha=0.3)

# ──── Panel C: Demo supply radar comparison ────
ax3 = fig.add_subplot(133, polar=True)
axes[2].set_visible(False)
ax3.set_position(axes[2].get_position())

dimensions = ['Task\nDiversity', 'Auto\nDemos', 'Real-World\nRelevance',
              'Few-Shot\nSupport', 'Scalability', 'Reproducibility']
N = len(dimensions)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

gym_scores = [2, 1, 1, 1, 2, 4]
rlb_scores = [5, 5, 5, 5, 5, 5]
gym_scores += gym_scores[:1]
rlb_scores += rlb_scores[:1]

ax3.plot(angles, gym_scores, 'o-', color='#90A4AE', linewidth=2, label='OpenAI Gym')
ax3.fill(angles, gym_scores, alpha=0.15, color='#90A4AE')
ax3.plot(angles, rlb_scores, 'o-', color='#EF5350', linewidth=2.5, label='RLBench')
ax3.fill(angles, rlb_scores, alpha=0.2, color='#EF5350')
ax3.set_xticks(angles[:-1])
ax3.set_xticklabels(dimensions, fontsize=8)
ax3.set_ylim(0, 5)
ax3.set_yticks([])
ax3.set_title('C) Benchmark\nCapability Profile', fontsize=11, pad=15)
ax3.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=8)

plt.tight_layout()
plt.savefig('rlbench_fig1_motivation.png', bbox_inches='tight')
plt.show()
print("RLBench dominates all dimensions — it is designed to be the one benchmark to rule them all.")

---
## Section 2 — Benchmark Design Properties

The authors of RLBench carefully designed the benchmark around 6 key properties. Understanding these helps explain every design decision in the paper.

### The 6 Properties

| Property | What It Means | Why It Matters |
|---|---|---|
| **Diversity** | 100 completely different tasks | Prevents overfitting; forces general algorithms |
| **Reproducibility** | All tasks in simulation | Every lab can run the exact same experiment |
| **Scale** | Infinite demonstrations per task | Deep learning needs lots of data |
| **Extensibility** | Easy to add new tasks via PyRep | Benchmark can grow over time with community help |
| **Tiered Difficulty** | Tasks range from simple reach to complex multi-step | Tests both beginners and state-of-the-art |
| **Realism** | Real robot model, lighting, domain randomisation | Helps transfer learning from simulation to real robot |

### The Hardware Setup

Every task in RLBench uses:
- **Robot:** Franka Emika Panda — a 7 Degree-of-Freedom (DoF) arm (can be swapped with one line of code)
- **Simulator:** V-REP (also called CoppeliaSim) + PyRep API
- **Cameras:** Two cameras (explained in Section 4)
- **Table:** Fixed wooden table with 3 directional lights

In [ ]:
# ─── Figure 2: Six Design Properties Visualisation ───────────────────────────

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14); ax.set_ylim(0, 7); ax.axis('off')
ax.set_title('Figure 2 — RLBench: Six Core Design Properties', fontsize=14, fontweight='bold')

properties = [
    # (x, y, title, body, color)
    (0.3,  3.8, '① Diversity',
     '100 completely unique\nhand-designed tasks\n(not just colour changes)',
     '#FFCDD2'),
    (2.65, 3.8, '② Reproducibility',
     'Fully in simulation\n(V-REP + PyRep)\nAny lab can run it',
     '#F8BBD9'),
    (5.0,  3.8, '③ Scale',
     'Infinite demos per task\nvia motion planning\n(no crowdsourcing needed)',
     '#E1BEE7'),
    (7.35, 3.8, '④ Extensibility',
     'Task builder tool\nPyRep API\nGitHub PR submission',
     '#BBDEFB'),
    (9.7,  3.8, '⑤ Tiered Difficulty',
     'Easy: reach target\nMedium: open door\nHard: empty dishwasher',
     '#C8E6C9'),
    (12.0, 3.8, '⑥ Realism',
     'Real robot model\nLighting & shadows\nDomain randomisation',
     '#FFF9C4'),
]

for (xc, yc, title, body, color) in properties:
    box = FancyBboxPatch((xc, yc - 1.5), 2.1, 3.2,
                         boxstyle='round,pad=0.12',
                         facecolor=color, edgecolor='#444', linewidth=1.5)
    ax.add_patch(box)
    ax.text(xc + 1.05, yc + 1.45, title, ha='center', va='top',
            fontsize=10, fontweight='bold')
    ax.text(xc + 1.05, yc + 0.85, body, ha='center', va='top',
            fontsize=8.5, linespacing=1.5)

# Central label at top
ax.text(7, 6.6, 'RLBench = Benchmark + Learning Environment + Task Creation Toolkit',
        ha='center', fontsize=11, color='#333',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFFDE7', edgecolor='#888'))

# Arrow connecting them
ax.annotate('', xy=(13.8, 5.2), xytext=(0.5, 5.2),
            arrowprops=dict(arrowstyle='->', color='#888', lw=1.5,
                            connectionstyle='arc3,rad=0'))
ax.text(7, 5.35, 'All 6 properties work together', ha='center', fontsize=9, color='#555')

plt.tight_layout()
plt.savefig('rlbench_fig2_properties.png', bbox_inches='tight')
plt.show()

---
## Section 3 — Core Concepts: Task, Variation, Episode

RLBench introduces a precise vocabulary with **3 key terms** that define how tasks are structured. Understanding this hierarchy is critical.

### The Hierarchy

```
TASK  (e.g., "stack_blocks")
│
├── Variation 0  → "stack ONE red block"
│     ├── Episode 0  (block at position A)
│     ├── Episode 1  (block at position B)
│     └── Episode E  (block at random position ...)
│
├── Variation 1  → "stack TWO red blocks"
│     ├── Episode 0
│     └── ...
│
└── Variation V  → "stack ONE maroon block"
      └── ...
```

### Formal Definitions

- **Episode trajectory** $\tau$: a sequence of (observation, action) pairs:
$$\tau = [(o_1, a_1), (o_2, a_2), \ldots, (o_T, a_T)]$$

- **Variation** $\nu$: a distribution over episodes — episodes are sampled as $\tau \sim \nu$

- **Task** $\mathfrak{T}$: a set of variations — $\mathfrak{T} = \{\nu_1, \nu_2, \ldots, \nu_N\}$

### Why "Variation" Exists
"Pick up the apple" and "Pick up the banana" — are these the same task or different tasks? This is philosophically tricky! RLBench solves it with *variations*: both are **variations** of the same **task** ("pick up the X"). Across variations, the **object or colour** changes. Across episodes of the same variation, only the **position** changes.

In [ ]:
# ─── Figure 3: Task / Variation / Episode Hierarchy ──────────────────────────

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14); ax.set_ylim(0, 7.5); ax.axis('off')
ax.set_title('Figure 3 — RLBench: Task → Variation → Episode Hierarchy',
             fontsize=13, fontweight='bold')

def draw_box(ax, x, y, w, h, text, color, fontsize=9, bold=False):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.08',
                         facecolor=color, edgecolor='#444', linewidth=1.3)
    ax.add_patch(box)
    weight = 'bold' if bold else 'normal'
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight=weight, wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# ── TASK level ──
draw_box(ax, 5.5, 6.3, 3.0, 0.9, 'TASK\n"stack_blocks"', '#FFCCBC', fontsize=11, bold=True)

# ── VARIATION level ──
variations = [
    (0.3,  4.4, 'Variation 0\n"Stack 1 red block"',   '#FFF9C4'),
    (4.1,  4.4, 'Variation 1\n"Stack 2 red blocks"',  '#FFF9C4'),
    (7.9,  4.4, 'Variation 2\n"Stack 3 red blocks"',  '#FFF9C4'),
    (11.5, 4.4, 'Variation V\n"Stack 1 maroon block"', '#FFF9C4'),
]
for (xv, yv, lab, col) in variations:
    draw_box(ax, xv, yv, 2.9, 1.1, lab, col, fontsize=8.5)
    draw_arrow(ax, 7.0, 6.3, xv + 1.45, yv + 1.1)

# ── EPISODE level (under variation 0) ──
episodes_v0 = [
    (0.3, 2.2, 'Episode 0\n(block at pos A)'),
    (1.5, 2.2, 'Episode 1\n(block at pos B)'),
    (2.7, 2.2, '...\n(random pos)'),
]
for (xe, ye, lab) in episodes_v0:
    draw_box(ax, xe, ye, 1.0, 1.0, lab, '#DCEDC8', fontsize=7.5)
    draw_arrow(ax, 1.75, 4.4, xe + 0.5, ye + 1.0)

# ── EPISODE level (under variation 1) ──
episodes_v1 = [
    (4.1, 2.2, 'Episode 0'),
    (5.1, 2.2, 'Episode 1'),
    (6.1, 2.2, '...'),
]
for (xe, ye, lab) in episodes_v1:
    draw_box(ax, xe, ye, 0.9, 1.0, lab, '#DCEDC8', fontsize=7.5)
    draw_arrow(ax, 5.55, 4.4, xe + 0.45, ye + 1.0)

# Ellipsis for other variations
ax.text(10.5, 4.95, '• • •', ha='center', fontsize=14, color='#888')

# ── Language descriptions ──
desc_texts = [
    '"stack one red block on the target"',
    '"place one of the red blocks on target"',
    '"build a tower out of one red block"',
]
ax.text(1.75, 1.5, 'Text descriptions per variation:', ha='center', fontsize=8.5,
        fontweight='bold', color='#1565C0')
for i, txt in enumerate(desc_texts):
    ax.text(1.75, 1.15 - i*0.35, txt, ha='center', fontsize=7.5, color='#1565C0', style='italic')

# ── Annotations ──
ax.annotate('Object / colour\nchanges across variations',
            xy=(7.0, 4.95), xytext=(9.5, 3.5),
            fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange'))

ax.annotate('Position\nchanges across episodes',
            xy=(2.0, 2.7), xytext=(4.2, 1.3),
            fontsize=9, color='darkgreen',
            arrowprops=dict(arrowstyle='->', color='darkgreen'))

plt.tight_layout()
plt.savefig('rlbench_fig3_hierarchy.png', bbox_inches='tight')
plt.show()

---
## Section 4 — Observation & Action Spaces

### What Does the Robot See?

RLBench provides **two cameras** giving different perspectives:

| Camera | Type | Position | What It Sees |
|---|---|---|---|
| **Over-the-shoulder stereo** | Stereo (2 lenses) | Fixed above & behind robot | Full workspace, task objects |
| **Eye-in-hand monocular** | Monocular (1 lens) | On the robot's wrist | Close-up of what the gripper is touching |

Each camera provides 3 types of images:
- **RGB** — colour image (what we normally see)
- **Depth** — distance from camera to each pixel (useful for 3-D reasoning)
- **Segmentation mask** — each object labelled with a different colour

Plus **proprioceptive** (body-sense) data:
- Joint angles (where each of the 7 arm joints is pointing)
- Joint velocities (how fast each joint is moving)
- Joint torques (how much force each joint is exerting)
- End-effector pose (position and orientation of the gripper)

### What Actions Can the Robot Take?

RLBench offers **7 different action spaces** — the user picks which one to use:

| Action Space | Description |
|---|---|
| Absolute/delta joint velocities | Control how fast each joint spins |
| Absolute/delta joint positions | Control the angle of each joint |
| Absolute/delta joint torques | Control force at each joint |
| Absolute/delta end-effector velocities | Control gripper speed |
| Absolute/delta end-effector poses | Control gripper position + orientation |

In [ ]:
# ─── Figure 4: Observation & Action Space ────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Figure 4 — RLBench Observation & Action Spaces', fontsize=13, fontweight='bold')

# ──── Panel A: Observation space diagram ────
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('A) Observation Space', fontsize=11)

# Camera boxes
cam_boxes = [
    (0.3, 5.5, 4.0, 1.8, 'Over-the-Shoulder Stereo Camera', '#BBDEFB'),
    (5.2, 5.5, 4.0, 1.8, 'Eye-in-Hand Monocular Camera', '#C8E6C9'),
]
obs_rgb    = ['RGB Image (128×128)', 'RGB Image (128×128)']
obs_depth  = ['Depth Image (128×128)', 'Depth Image (128×128)']
obs_seg    = ['Seg. Mask (128×128)', 'Seg. Mask (128×128)']

for i, (xc, yc, w, h, title, col) in enumerate(cam_boxes):
    box = FancyBboxPatch((xc, yc), w, h, boxstyle='round,pad=0.1',
                         facecolor=col, edgecolor='#444', linewidth=1.3)
    ax.add_patch(box)
    ax.text(xc + w/2, yc + h - 0.3, title, ha='center', fontsize=9, fontweight='bold')

    # Sub-observations
    for j, (lab, col2) in enumerate([(obs_rgb[i], '#E3F2FD'),
                                       (obs_depth[i], '#F3E5F5'),
                                       (obs_seg[i], '#FFF8E1')]):
        sub = FancyBboxPatch((xc + 0.15, yc + 0.12 + j*0.42), w - 0.3, 0.38,
                             boxstyle='round,pad=0.04', facecolor=col2,
                             edgecolor='#888', linewidth=0.8)
        ax.add_patch(sub)
        ax.text(xc + w/2, yc + 0.31 + j*0.42, lab, ha='center', fontsize=8)

# Proprioception box
prop_box = FancyBboxPatch((2.0, 3.2), 5.5, 1.8,
                           boxstyle='round,pad=0.1',
                           facecolor='#FFE0B2', edgecolor='#444', linewidth=1.3)
ax.add_patch(prop_box)
ax.text(4.75, 4.75, 'Proprioceptive Data (Robot Body Sense)', ha='center', fontsize=9, fontweight='bold')
for j, lab in enumerate(['Joint angles (7 DoF)', 'Joint velocities', 'Joint torques', 'End-effector pose']):
    ax.text(4.75, 4.38 - j*0.32, f'• {lab}', ha='center', fontsize=8)

# Arrows downward to VLA box
ax.annotate('', xy=(4.75, 2.0), xytext=(2.3, 3.2),
            arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
ax.annotate('', xy=(4.75, 2.0), xytext=(7.2, 3.2),
            arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))
ax.annotate('', xy=(4.75, 2.0), xytext=(4.75, 3.2),
            arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

agent_box = FancyBboxPatch((3.0, 0.8), 3.5, 1.1,
                            boxstyle='round,pad=0.1',
                            facecolor='#EDE7F6', edgecolor='#333', linewidth=2)
ax.add_patch(agent_box)
ax.text(4.75, 1.35, 'Robot Agent / Policy', ha='center', fontsize=10, fontweight='bold')
ax.text(4.75, 0.55, '→ outputs action', ha='center', fontsize=9, color='#6A1B9A')

# ──── Panel B: Action space bar chart ────
ax2 = axes[1]
action_spaces = [
    'Abs. Joint\nVelocities',
    'Delta Joint\nVelocities',
    'Abs. Joint\nPositions',
    'Delta Joint\nPositions',
    'Abs. EEF\nVelocities',
    'Delta EEF\nPoses',
    'Abs. EEF\nPoses',
]
dim = [7, 7, 7, 7, 6, 7, 7]   # Dimensionality (7 joints or 6-DOF EEF)
use_cases = ['RL', 'RL', 'IL', 'IL', 'IL', 'IL', 'IL']
bar_colors = ['#EF9A9A' if u == 'RL' else '#90CAF9' for u in use_cases]

bars = ax2.barh(action_spaces, dim, color=bar_colors, edgecolor='black', linewidth=0.7)
ax2.set_xlabel('Action Dimensionality', fontsize=10)
ax2.set_title('B) Available Action Spaces\n(RLBench offers 7 choices)', fontsize=11)
for bar in bars:
    ax2.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             str(int(bar.get_width())), va='center', fontsize=9)

rl_patch = mpatches.Patch(color='#EF9A9A', label='Suits RL')
il_patch = mpatches.Patch(color='#90CAF9', label='Suits IL / Planning')
ax2.legend(handles=[rl_patch, il_patch], fontsize=9)
ax2.set_xlim(0, 9); ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('rlbench_fig4_obs_action.png', bbox_inches='tight')
plt.show()
print("6 modalities total: 2 cameras × 3 image types + proprioception. Researcher picks the action space.")

---
## Section 5 — Demonstrations: Infinite Free Training Data

One of RLBench's biggest contributions is the ability to generate **infinite demonstrations** automatically — no human labour required.

### How It Works

When a task creator builds a new task, they define a set of **waypoints** — key positions the robot must reach in order to complete the task. For example, for "open the door":
1. Move hand to door handle position
2. Grasp the handle
3. Rotate wrist to turn the handle
4. Pull arm to swing the door open

The **Open Motion Planning Library (OMPL)** then automatically plans a collision-free path between these waypoints. This path is the demonstration.

Because the positions of objects are **randomised** at the start of each episode, each episode produces a **different** demonstration, giving infinite variety.

### Formal Definition

The expert policy $\pi^*$ generates demonstration trajectories:
$$\tau^* = \pi^*(\text{waypoints}, \text{scene state}) \rightarrow [(o_1, a_1^*), \ldots, (o_T, a_T^*)]$$

These demonstrations can be used for:
- **Imitation learning** — train a policy to copy the demos
- **Bootstrapping RL** — pre-train a policy before fine-tuning with rewards
- **Few-shot learning** — show a robot $K$ examples of a new task

In [ ]:
# ─── Figure 5: Demo Generation Pipeline + Task Horizon Distribution ──────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 5 — Demonstration Generation in RLBench', fontsize=13, fontweight='bold')

# ──── Panel A: Demo generation pipeline ────
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 7); ax.axis('off')
ax.set_title('A) How Infinite Demos Are Generated', fontsize=11)

pipeline = [
    (5.0, 6.2, 'Task Definition\n(.ttm + .py files)', '#FFF9C4'),
    (5.0, 4.5, 'Waypoints\n(key positions in task)', '#BBDEFB'),
    (5.0, 2.8, 'OMPL Motion Planner\n(finds collision-free path)', '#C8E6C9'),
    (5.0, 1.1, 'Demo Trajectory τ*\n[(o₁,a₁*), ..., (oT,aT*)]', '#FFE0B2'),
]
for (xc, yc, lab, col) in pipeline:
    box = FancyBboxPatch((xc-2.5, yc-0.55), 5.0, 1.1,
                         boxstyle='round,pad=0.1',
                         facecolor=col, edgecolor='#444', linewidth=1.3)
    ax.add_patch(box)
    ax.text(xc, yc, lab, ha='center', va='center', fontsize=9)

for i in range(len(pipeline)-1):
    _, y1, _, _ = pipeline[i]
    _, y2, _, _ = pipeline[i+1]
    ax.annotate('', xy=(5.0, y2 + 0.55), xytext=(5.0, y1 - 0.55),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))

# Randomisation annotation
ax.annotate('Object positions\nrandomised → infinite demos!',
            xy=(5.0, 4.5), xytext=(7.5, 5.2),
            fontsize=8.5, color='darkgreen',
            arrowprops=dict(arrowstyle='->', color='darkgreen'))

ax.text(5.0, 0.25, 'Used for: Imitation Learning, RL bootstrapping, Few-Shot Learning',
        ha='center', fontsize=8, color='#555', style='italic')

# ──── Panel B: Task horizon distribution (modelled from paper Fig 7) ────
ax2 = axes[1]

# Simulate task length distribution similar to paper's Fig 7
np.random.seed(42)
n_tasks = 75
# Exponential-like distribution: most tasks short, some very long
short_tasks  = np.random.randint(80,  200, 30)   # simple tasks
medium_tasks = np.random.randint(200, 500, 30)   # medium tasks
long_tasks   = np.random.randint(500, 1000, 15)  # long-horizon tasks
all_lengths  = np.concatenate([short_tasks, medium_tasks, long_tasks])
task_indices = np.arange(n_tasks)

# Sort by length for clearer visualisation
all_lengths_sorted = np.sort(all_lengths)[::-1]

# Colour by tier
bar_cols = []
for v in all_lengths_sorted:
    if v < 200:   bar_cols.append('#A5D6A7')
    elif v < 500: bar_cols.append('#FFE082')
    else:         bar_cols.append('#EF9A9A')

ax2.bar(task_indices, all_lengths_sorted, color=bar_cols, edgecolor='none', width=1.0)

# Threshold lines
ax2.axhline(y=200, color='#388E3C', linestyle='--', linewidth=1.5, label='Easy threshold (< 200 steps)')
ax2.axhline(y=500, color='#F57F17', linestyle='--', linewidth=1.5, label='Medium threshold (< 500 steps)')

easy_patch   = mpatches.Patch(color='#A5D6A7', label='Easy tasks')
medium_patch = mpatches.Patch(color='#FFE082', label='Medium tasks')
hard_patch   = mpatches.Patch(color='#EF9A9A', label='Hard tasks (long horizon)')
ax2.legend(handles=[easy_patch, medium_patch, hard_patch], fontsize=8, loc='upper right')

ax2.set_xlabel('Task Index (sorted by length)', fontsize=10)
ax2.set_ylabel('Episode Length (timesteps)', fontsize=10)
ax2.set_title('B) Task Horizon Distribution\n(75-task sample, 100–1000 timesteps)', fontsize=11)
ax2.grid(axis='y', alpha=0.3)

# Annotate long task
ax2.annotate('"Empty dishwasher":\nopen door → slide tray →\ngrasp plate → lift out',
             xy=(2, 900), xytext=(15, 750),
             fontsize=8, color='darkred',
             arrowprops=dict(arrowstyle='->', color='darkred'))

plt.tight_layout()
plt.savefig('rlbench_fig5_demos.png', bbox_inches='tight')
plt.show()
print("Key point: Task lengths range from 100 to 1000 timesteps — a 10× range in difficulty!")

---
## Section 6 — The Environment API

RLBench exposes a clean Python API modelled after the **standard reinforcement learning agent-environment loop**. If you have ever used OpenAI Gym, this will look very familiar.

### The Standard RL Loop

```
env.reset() → initial observation
     ↓
agent.act(observation) → action
     ↓
env.step(action) → next observation, reward, done
     ↓
  repeat until done
```

### Reward

RLBench uses **completely sparse rewards**:
$$r_t = \begin{cases} +1 & \text{if task fully completed at step } t \\ 0 & \text{otherwise} \end{cases}$$

This is intentionally hard — the robot gets no partial credit for being "close" to the goal. This forces algorithms to solve the full task rather than exploiting partial rewards.

### The Code Structure (from the paper)

```python
from rlbench.environment import Environment
from rlbench.action_modes import ActionMode

env = Environment(DATASET, ActionMode.ABS_JOINT_VELOCITY)
env.launch()

task = env.sample_task()
demos = task.get_demos(2)       # get 2 demo trajectories

agent = Agent()
agent.ingest(demos)             # learn from demos

for i in range(training_steps):
    if i % episode_length == 0:
        descriptions, obs = task.reset()   # start new episode
    action = agent.act(obs)               # agent decides action
    obs, reward, terminate = task.step(action)  # environment responds

env.shutdown()
```

In [ ]:
# ─── Figure 6: RL Loop Simulation & Sparse Reward Illustration ───────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 6 — The RLBench Environment API & Sparse Rewards', fontsize=13, fontweight='bold')

# ──── Panel A: RL loop diagram ────
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('A) Agent–Environment Loop', fontsize=11)

# Environment box
env_box = FancyBboxPatch((5.5, 3.0), 3.8, 2.5,
                          boxstyle='round,pad=0.15',
                          facecolor='#BBDEFB', edgecolor='#1565C0', linewidth=2)
ax.add_patch(env_box)
ax.text(7.4, 5.15, 'ENVIRONMENT', ha='center', fontsize=10, fontweight='bold', color='#1565C0')
ax.text(7.4, 4.7,  'TaskEnvironment', ha='center', fontsize=9)
ax.text(7.4, 4.35, '• Simulated lab scene', ha='center', fontsize=8)
ax.text(7.4, 4.0,  '• task.step(action)', ha='center', fontsize=8)
ax.text(7.4, 3.65, '• Reward: sparse {0, +1}', ha='center', fontsize=8)

# Agent box
agent_box = FancyBboxPatch((0.5, 3.0), 3.8, 2.5,
                            boxstyle='round,pad=0.15',
                            facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(agent_box)
ax.text(2.4, 5.15, 'AGENT', ha='center', fontsize=10, fontweight='bold', color='#2E7D32')
ax.text(2.4, 4.7,  'Policy π', ha='center', fontsize=9)
ax.text(2.4, 4.35, '• Observes o_t', ha='center', fontsize=8)
ax.text(2.4, 4.0,  '• Chooses action a_t', ha='center', fontsize=8)
ax.text(2.4, 3.65, '• Learns from reward', ha='center', fontsize=8)

# Arrows: action (top), observation (bottom)
ax.annotate('', xy=(5.5, 5.1), xytext=(4.3, 5.1),
            arrowprops=dict(arrowstyle='->', color='#E53935', lw=2.5))
ax.text(4.9, 5.35, 'action aₜ', ha='center', fontsize=9, color='#E53935', fontweight='bold')

ax.annotate('', xy=(4.3, 3.5), xytext=(5.5, 3.5),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2.5))
ax.text(4.9, 3.25, 'obs oₜ₊₁, reward rₜ, done', ha='center', fontsize=9,
        color='#1565C0', fontweight='bold')

# Reset arrow
ax.annotate('task.reset()  →  descriptions + obs₀',
            xy=(2.4, 6.8), xytext=(2.4, 5.5),
            ha='center', fontsize=9, color='#555',
            arrowprops=dict(arrowstyle='->', color='#555'))

# Text description box
desc_box = FancyBboxPatch((0.5, 1.5), 8.8, 1.2,
                           boxstyle='round,pad=0.1',
                           facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=1.5)
ax.add_patch(desc_box)
ax.text(4.9, 2.4, 'Language Descriptions (from variation)',
        ha='center', fontsize=9, fontweight='bold', color='#E65100')
ax.text(4.9, 1.95, '"stack one red block on the target"   |   "place one red block on the target"',
        ha='center', fontsize=8.5, color='#E65100', style='italic')

# ──── Panel B: Sparse reward over an episode ────
ax2 = axes[1]
T = 120  # episode length
t = np.arange(T)

# Successful episode: reward = 0 until step 98, then 1
reward_success = np.zeros(T)
reward_success[98:] = 1.0

# Failed episode: always 0
reward_fail = np.zeros(T)

ax2.step(t, reward_success, where='post', color='#4CAF50', linewidth=2.5, label='Successful episode')
ax2.step(t, reward_fail - 0.02, where='post', color='#F44336', linewidth=2,
         linestyle='--', label='Failed episode')

# Shade the non-reward region
ax2.fill_between(t, reward_success, alpha=0.1, color='#4CAF50', step='post')

ax2.axvline(x=98, color='#388E3C', linestyle=':', linewidth=1.5)
ax2.text(99, 0.5, 'Task\ncompleted!', fontsize=9, color='#388E3C')
ax2.text(40, 0.35, 'NO reward during task execution\n(completely sparse)',
         fontsize=9, color='#555', style='italic',
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax2.set_xlabel('Timestep t', fontsize=10)
ax2.set_ylabel('Reward rₜ', fontsize=10)
ax2.set_title('B) Sparse Reward Signal\n(+1 only on task completion)', fontsize=11)
ax2.set_ylim(-0.2, 1.3)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('rlbench_fig6_api.png', bbox_inches='tight')
plt.show()
print("Sparse reward is intentionally hard — it forces the algorithm to actually solve the task, not exploit partial reward.")

---
## Section 7 — The Task Builder: How to Create New Tasks

One of RLBench's most powerful features is that **anyone can add new tasks**. The task builder makes this as easy as possible.

### Two Files Per Task

Each task requires exactly two files:

| File | Format | Purpose |
|---|---|---|
| Scene file | `.ttm` (V-REP model) | 3D objects, their positions, demo waypoints |
| Code file | `.py` (Python) | Wires scene to RLBench backend, defines success condition |

### The Three Lifecycle Functions

Every task `.py` file must implement three functions:

```python
init_task()           # Called once: register graspable objects, success conditions
init_variation(i)     # Called per variation: apply object changes, return descriptions
init_episode()        # Called per episode: randomise positions
```

### Example Task: Take Lid Off Saucepan (from the paper)

```python
class TakeLidOffSaucepan(Task):
    def init_task(self):
        lid = Shape('saucepan_lid')
        success_detector = ProximitySensor('success')
        self.register_graspable_objects([lid])
        cond_set = [
            GraspedCondition(self.robot.gripper, lid),   # must be holding lid
            DetectedCondition(lid, success_detector)    # lid must be above saucepan
        ]
        self.register_success_conditions([cond_set])

    def init_episode(self, index):
        return ['take lid off the saucepan']   # language description

    def variation_count(self):
        return 1
```

Once created, users submit via **GitHub pull request** to the growing task repository.

In [ ]:
# ─── Figure 7: Task Builder Lifecycle + Word Frequency ───────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Figure 7 — Task Builder: How New Tasks Are Created', fontsize=13, fontweight='bold')

# ──── Panel A: Task lifecycle diagram ────
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 9); ax.axis('off')
ax.set_title('A) Task File Lifecycle', fontsize=11)

steps_lc = [
    (5.0, 8.2, 'User creates .ttm (V-REP scene)\n+ .py (task code)', '#FFF9C4', 1.8),
    (5.0, 6.5, 'init_task()\nRegister objects & success conditions', '#BBDEFB', 1.8),
    (5.0, 4.8, 'init_variation(i)\nApply colour/object changes\nReturn text descriptions', '#C8E6C9', 2.2),
    (5.0, 2.9, 'init_episode()\nRandomise object positions', '#FFE0B2', 1.5),
    (5.0, 1.3, 'Task Validation Tool\n(auto-checks planning succeeds)', '#E1BEE7', 1.3),
    (5.0, 0.0, 'GitHub Pull Request\n→ Community task repository', '#FFCCBC', 1.0),
]

call_counts = ['Once', 'Per variation', 'Per episode', '', '']

for i, (xc, yc, lab, col, height) in enumerate(steps_lc):
    half_h = height / 2
    box = FancyBboxPatch((xc - 3.2, yc - half_h), 6.4, height,
                         boxstyle='round,pad=0.1',
                         facecolor=col, edgecolor='#444', linewidth=1.3)
    ax.add_patch(box)
    ax.text(xc, yc, lab, ha='center', va='center', fontsize=8.5)
    if i < len(call_counts) and call_counts[i]:
        ax.text(8.5, yc, call_counts[i], ha='center', va='center',
                fontsize=8, color='#1565C0', style='italic')

    if i < len(steps_lc) - 1:
        next_yc = steps_lc[i+1][1]
        next_h  = steps_lc[i+1][4]
        ax.annotate('', xy=(xc, next_yc + next_h/2),
                    xytext=(xc, yc - height/2),
                    arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# ──── Panel B: Word frequency in task descriptions ────
ax2 = axes[1]
# Most common action words in RLBench task descriptions (representative)
words = ['open', 'put', 'take', 'close', 'pick', 'place', 'slide', 'push',
         'turn', 'stack', 'grab', 'insert', 'remove', 'lift', 'move']
# Approximate frequencies (exponential-like distribution, as seen in paper Fig 7)
freqs = [48, 42, 35, 30, 28, 25, 20, 18, 15, 12, 10, 9, 7, 6, 5]

bar_cols2 = plt.cm.Blues(np.linspace(0.4, 0.9, len(words)))[::-1]
bars2 = ax2.bar(words, freqs, color=bar_cols2, edgecolor='black', linewidth=0.7)
ax2.set_xlabel('Action Word', fontsize=10)
ax2.set_ylabel('Frequency in Task Descriptions', fontsize=10)
ax2.set_title('B) Most Common Action Words\nin RLBench Task Descriptions', fontsize=11)
ax2.tick_params(axis='x', rotation=45, labelsize=9)
ax2.grid(axis='y', alpha=0.3)

# Annotate top words
for bar, val in zip(bars2[:5], freqs[:5]):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.5, str(val),
             ha='center', fontsize=9, fontweight='bold')

ax2.text(0.6, 0.85,
         'Top words reflect\nhousehold robot tasks',
         transform=ax2.transAxes, fontsize=9,
         bbox=dict(boxstyle='round', facecolor='#FFF9C4', alpha=0.9))

plt.tight_layout()
plt.savefig('rlbench_fig7_taskbuilder.png', bbox_inches='tight')
plt.show()

---
## Section 8 — The Few-Shot Challenge (v1.0)

The **centrepiece** of RLBench is its few-shot learning challenge — the first large-scale few-shot benchmark in robotics.

### The Core Idea

Humans learn new tasks very quickly after seeing just a few examples. Can robots do the same?

> *Given K demonstrations of an unseen task, can the robot perform that task in new configurations?*

### Challenge Setup

| Split | Tasks | Used For |
|---|---|---|
| **Meta-train** | 90 tasks (90%) | Training — can use in any way |
| **Meta-test** | 10 tasks (10%) | Evaluation — system has never seen these |

### Evaluation Protocol

At test time, the system receives **K demonstrations** of each unseen task. It must then perform the task in new configurations. Results are reported for:
- **1-shot** (K=1): only 1 demo shown
- **5-shot** (K=5): 5 demos shown
- **20-shot** (K=20): 20 demos shown

### Why This Is Hard

Prior few-shot work treated variations of the same task ("pick up apple" vs "pick up banana") as different tasks — a very narrow definition. RLBench demands genuine generalisation: the 10 test tasks are **completely different** from the 90 training tasks.

### Few-Shot Methods Compared

| Method Type | Examples | Core Idea |
|---|---|---|
| Recurrent/memory | LSTM-based, RL² | Use memory to adapt from demos |
| Metric learning | Matching Nets, Prototypical Nets | Compare test input to demo embeddings |
| Gradient-based | MAML, LEO | Learn how to learn from few examples |

In [ ]:
# ─── Figure 8: Few-Shot Challenge Setup & K-Shot Learning Curve ──────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 8 — The RLBench Few-Shot Challenge (v1.0)', fontsize=13, fontweight='bold')

# ──── Panel A: Train/test split diagram ────
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 8); ax.axis('off')
ax.set_title('A) Challenge Setup: Train-Test Split', fontsize=11)

# 100 tasks as grid
n_cols = 10
for idx in range(100):
    row = idx // n_cols
    col = idx  % n_cols
    x = 0.3 + col * 0.92
    y = 6.8 - row * 0.62
    color = '#EF5350' if idx < 10 else '#66BB6A'  # first 10 = test
    rect = FancyBboxPatch((x, y), 0.78, 0.5, boxstyle='round,pad=0.03',
                          facecolor=color, edgecolor='white', linewidth=0.8, alpha=0.85)
    ax.add_patch(rect)

# Labels
train_patch = mpatches.Patch(color='#66BB6A', label='Meta-Train (90 tasks, 90%)')
test_patch  = mpatches.Patch(color='#EF5350', label='Meta-Test (10 tasks, 10%)')
ax.legend(handles=[train_patch, test_patch], fontsize=9, loc='lower left')
ax.text(5.0, 7.6, '100 RLBench Tasks', ha='center', fontsize=11, fontweight='bold')

# Annotation
ax.annotate('Test tasks: completely\nDIFFERENT from training!',
            xy=(1.5, 6.5), xytext=(3.0, 5.0),
            fontsize=9, color='darkred',
            arrowprops=dict(arrowstyle='->', color='darkred'))

# K-shot arrow
ax.text(5.0, 0.6,
        'At test time: given K demos of unseen task → evaluate in new episode configs',
        ha='center', fontsize=9, color='#1565C0',
        bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.9))

# ──── Panel B: K-shot performance curves ────
ax2 = axes[1]
K_vals = [1, 2, 5, 10, 20, 50]

# Simulated performance curves for 3 method types
# Modelled from expected behaviour described in paper
recurrent_sr  = [12,  18,  30,  42,  55,  65]
metric_sr     = [8,   14,  24,  35,  46,  58]
gradient_sr   = [15,  22,  35,  48,  60,  70]
human_sr      = [85,  88,  90,  92,  93,  94]  # reference

ax2.semilogx(K_vals, recurrent_sr,  'o-', color='#42A5F5', linewidth=2.5, label='Recurrent (RL²-style)')
ax2.semilogx(K_vals, metric_sr,     's-', color='#EF5350', linewidth=2.5, label='Metric Learning')
ax2.semilogx(K_vals, gradient_sr,   '^-', color='#66BB6A', linewidth=2.5, label='Gradient-Based (MAML-style)')
ax2.semilogx(K_vals, human_sr,      'k--', linewidth=1.5, alpha=0.6, label='Human Performance (ref.)')

# Mark 1-shot, 5-shot, 20-shot evaluation points
for k_mark, col in [(1, '#E53935'), (5, '#F57F17'), (20, '#2E7D32')]:
    ax2.axvline(x=k_mark, color=col, linestyle=':', linewidth=1.5)
    ax2.text(k_mark*1.05, 2, f'{k_mark}-shot', color=col, fontsize=9)

ax2.set_xlabel('K (number of demonstrations)', fontsize=10)
ax2.set_ylabel('Task Success Rate (%)', fontsize=10)
ax2.set_title('B) Expected K-Shot Performance\n(Illustrative — paper proposes challenge, not results)',
              fontsize=10)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)
ax2.set_ylim(0, 100)

ax2.fill_between(K_vals, gradient_sr, human_sr,
                 alpha=0.1, color='gray', label='Gap to close')
ax2.text(15, 77, 'Gap to\nhuman performance', fontsize=8.5, color='gray')

plt.tight_layout()
plt.savefig('rlbench_fig8_fewshot.png', bbox_inches='tight')
plt.show()
print("RLBench's few-shot challenge v1.0: report 1-shot, 5-shot, and 20-shot results.")

---
## Section 9 — Other Research Applications

Beyond the few-shot challenge, RLBench supports 5 other research directions:

### a) Reinforcement Learning
Unlike OpenAI Gym's toy tasks, RLBench provides **real-world manipulation scenarios** that actually resemble what household robots will do. The large supply of demos also enables **demo-bootstrapped RL** — starting RL from a good initial policy instead of random exploration.

### b) Imitation Learning
Most IL papers define their own tasks, making comparison impossible. RLBench provides a **standard set of demos** (or unlimited on-the-fly generation) so all papers are tested on the same tasks.

### c) Sim-to-Real Transfer
The simulated Franka Panda can be **swapped with one line of code** for whatever arm a lab has. Domain randomisation is built in (object textures, lighting). This makes RLBench directly useful for labs working on transferring simulation-trained policies to real hardware.

### d) Multi-Task Learning
In contrast to few-shot (generalise to *new* tasks), multi-task learning tries to master *all* training tasks simultaneously. RLBench's 100 tasks provide a rich setting for this.

### e) SLAM
Simultaneous Localisation and Mapping — building a map of an environment while navigating it. RLBench opens the question: what kind of map representation (sparse, dense, semi-dense) is best for helping a robot manipulate objects?

In [ ]:
# ─── Figure 9: Research Application Overview ─────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 6.5))
ax.set_xlim(0, 14); ax.set_ylim(0, 7); ax.axis('off')
ax.set_title('Figure 9 — RLBench: Five Research Application Domains', fontsize=13, fontweight='bold')

# Central hub
hub = Circle((7, 3.5), 1.35, facecolor='#EF5350', edgecolor='#B71C1C', linewidth=2.5)
ax.add_patch(hub)
ax.text(7, 3.6, 'RLBench', ha='center', va='center', fontsize=12,
        fontweight='bold', color='white')
ax.text(7, 3.1, '100 Tasks', ha='center', va='center', fontsize=9, color='#FFCDD2')

# Spoke nodes
spokes = [
    (2.0, 5.8, 'a) Reinforcement\nLearning',
     'Sparse reward\nDemo-bootstrapped RL\nPartial observability',
     '#BBDEFB', '#1565C0'),
    (11.5, 5.8, 'b) Imitation\nLearning',
     'Standard task set\nInfinite demo gen\nReproducible results',
     '#C8E6C9', '#2E7D32'),
    (1.0, 1.5, 'c) Sim-to-Real\nTransfer',
     'Swap robot arm\nin 1 line of code\nDomain randomisation',
     '#FFF9C4', '#F57F17'),
    (12.5, 1.5, 'd) Multi-Task\nLearning',
     'Learn all 100 tasks\nSimultaneously\nGeneralise within tasks',
     '#E1BEE7', '#6A1B9A'),
    (7.0, 0.2, 'e) SLAM\n(Mapping)',
     'Task-oriented mapping\nSparse vs dense map\nManipulation + localisation',
     '#FFE0B2', '#E65100'),
]

for (xc, yc, title, body, face_col, text_col) in spokes:
    box = FancyBboxPatch((xc-1.8, yc-1.1), 3.6, 2.2,
                         boxstyle='round,pad=0.1',
                         facecolor=face_col, edgecolor=text_col, linewidth=1.5)
    ax.add_patch(box)
    ax.text(xc, yc + 0.65, title, ha='center', va='center',
            fontsize=9.5, fontweight='bold', color=text_col)
    ax.text(xc, yc - 0.2, body, ha='center', va='center',
            fontsize=8, color='#333', linespacing=1.4)

    # Arrow from hub to spoke
    dx = xc - 7; dy = yc - 3.5
    dist = np.sqrt(dx**2 + dy**2)
    nx, ny = dx/dist, dy/dist
    x_start = 7 + nx * 1.35
    y_start = 3.5 + ny * 1.35
    x_end   = xc - nx * 1.85
    y_end   = yc - ny * 1.15
    ax.annotate('', xy=(x_end, y_end), xytext=(x_start, y_start),
                arrowprops=dict(arrowstyle='->', color='#888', lw=1.8))

plt.tight_layout()
plt.savefig('rlbench_fig9_applications.png', bbox_inches='tight')
plt.show()

---
## Section 10 — Comparing RLBench with AutoBio

Having covered both RLBench (2019) and AutoBio (2025) in this tutorial series, it is useful to compare them directly.

| Feature | RLBench (2019) | AutoBio (2025) |
|---|---|---|
| Domain | Household manipulation | Biology laboratory |
| Number of tasks | 100 | ~20 (3 tiers) |
| Physics engine | V-REP + PyRep | MuJoCo + custom plugins |
| Custom physics | Standard rigid body | Thread, Detent, Eccentric, Liquid |
| Rendering | Good (lighting, shadows) | PBR (transparent glass, LCD UI) |
| Demo generation | Motion planning (OMPL) | Scripted oracle policy |
| Precision requirement | Low-medium | Very high (<1 mm) |
| Transparency challenge | No | Yes (glass, liquids) |
| Digital UI interaction | No | Yes (LCD screens) |
| Few-shot challenge | Yes (v1.0) | No (evaluation only) |
| VLA model integration | Partial | Full (OpenVLA, RDT-1B) |
| Year | 2019 | 2025 |

In [ ]:
# ─── Figure 10: RLBench vs AutoBio radar comparison ──────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Figure 10 — RLBench vs AutoBio: A 6-Year Progression',
             fontsize=13, fontweight='bold')

# ──── Panel A: Radar ────
ax = fig.add_subplot(121, polar=True)
axes[0].set_visible(False)
ax.set_position(axes[0].get_position())

dimensions = [
    'Task\nBreadth',
    'Demo\nGeneration',
    'Precision\nRequired',
    'Rendering\nFidelity',
    'Few-Shot\nSupport',
    'Physics\nRealism',
]
N = len(dimensions)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

rlb_scores  = [5, 5, 2, 3, 5, 3]
auto_scores = [2, 3, 5, 5, 1, 5]
rlb_scores  += rlb_scores[:1]
auto_scores += auto_scores[:1]

ax.plot(angles, rlb_scores,  'o-', color='#1976D2', linewidth=2.5, label='RLBench (2019)')
ax.fill(angles, rlb_scores,  alpha=0.2, color='#1976D2')
ax.plot(angles, auto_scores, 'o-', color='#E91E63', linewidth=2.5, label='AutoBio (2025)')
ax.fill(angles, auto_scores, alpha=0.2, color='#E91E63')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dimensions, fontsize=9)
ax.set_ylim(0, 5); ax.set_yticks([])
ax.set_title('A) Capability Comparison\nRLBench vs AutoBio', fontsize=11, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.15), fontsize=9)

# ──── Panel B: Timeline of progress ────
ax2 = axes[1]
ax2.set_xlim(2017, 2028); ax2.set_ylim(-1, 10); ax2.axis('off')
ax2.set_title('B) Robot Learning Benchmark Timeline', fontsize=11)

events = [
    (2017, 4.0, 'Meta-World\n(50 tasks)', '#90A4AE'),
    (2018, 6.5, 'RoboTurk\n(3 tasks, crowdsourced)', '#90A4AE'),
    (2019, 8.5, 'RLBench\n(100 tasks,\nauto-demos)', '#1976D2'),
    (2020, 2.5, 'Simitate\n(hybrid real/sim)', '#90A4AE'),
    (2023, 5.5, 'ManiSkill2\n(generalisable skills)', '#64B5F6'),
    (2024, 3.5, 'Chemistry3D\n(chem lab)', '#FFB74D'),
    (2025, 7.5, 'AutoBio\n(bio lab, VLA,\nPBR rendering)', '#E91E63'),
]

# Timeline axis
ax2.axhline(y=0.5, color='#333', linewidth=2, xmin=0.02, xmax=0.98)
for year in range(2017, 2027):
    ax2.axvline(x=year, ymin=0.04, ymax=0.08, color='#333', linewidth=1.2)
    ax2.text(year, -0.5, str(year), ha='center', fontsize=8, color='#333')

for (year, y, label, col) in events:
    dot = Circle((year, 0.5), 0.12, facecolor=col, edgecolor='black', linewidth=1.2, zorder=5)
    ax2.add_patch(dot)
    ax2.plot([year, year], [0.5, y - 0.8], color=col, linewidth=1.2, linestyle='--', alpha=0.7)
    box = FancyBboxPatch((year - 1.0, y - 0.7), 2.0, 1.4,
                         boxstyle='round,pad=0.1',
                         facecolor=col, alpha=0.85,
                         edgecolor='black', linewidth=1)
    ax2.add_patch(box)
    ax2.text(year, y, label, ha='center', va='center',
             fontsize=7.5, color='white' if col in ['#1976D2', '#E91E63'] else '#222')

plt.tight_layout()
plt.savefig('rlbench_fig10_comparison.png', bbox_inches='tight')
plt.show()
print("RLBench (2019) and AutoBio (2025) are complementary: breadth vs depth, household vs lab.")

---
## Section 11 — Summary & Key Takeaways

### What RLBench Contributes

1. **One unified benchmark** for many robot learning paradigms (RL, IL, few-shot, multi-task, sim-to-real, SLAM)
2. **100 unique tasks** that are genuinely diverse and range from simple to complex
3. **Infinite demonstrations** generated automatically — no human labelling required
4. **The first large-scale few-shot challenge** in robotics (1-shot, 5-shot, 20-shot)
5. **An open task-builder tool** so the community can grow the benchmark over time

### What Makes It Different

- Existing benchmarks were either **too simple** (OpenAI Gym) or **too narrow** (RoboTurk: 3 tasks)
- RLBench is the first to serve **both classical and deep-learning** methods on an even playing field
- It is designed to **grow** — versioned challenges (v1.0, v2.0...) keep results meaningful as tasks are added

### Open Research Questions (from the paper)

- Can current RL algorithms solve long-horizon tasks (1000 timesteps) with sparse rewards?
- What is the best K-shot method for genuine task generalisation in robotics?
- How well do policies trained in RLBench transfer to a real Franka arm?
- Can SLAM maps usefully guide robot manipulation?
- What multi-task architecture enables one robot to master all 100 tasks?

### References

- James et al. (2019). *RLBench: The Robot Learning Benchmark & Learning Environment*. arXiv:1909.12271
- Brockman et al. (2016). *OpenAI Gym*. arXiv:1606.01540
- Yu et al. (2019). *Meta-World: A Benchmark and Evaluation for Multi-Task and Meta Reinforcement Learning*
- James et al. (2019). *PyRep: Bringing V-REP to Deep Robot Learning*. arXiv:1906.11176
- Sucan et al. (2012). *The Open Motion Planning Library (OMPL)*. IEEE Robotics & Automation Magazine

---
## Section 12 — Exercises

Work through these exercises to test your understanding.

---

### Exercise 1 — Task/Variation/Episode Counts

The `stack_blocks` task has **V = 4 variations** (1 red, 2 red, 3 red, 1 maroon). Each variation has **E = 50 episodes**.

a) How many total episodes does this task have?  
b) If each episode has **T = 200 timesteps**, how many total timesteps of data does this one task provide?  
c) If all 100 tasks have the same configuration, how many total timesteps does the entire RLBench benchmark supply?  
d) At 30 frames per second, how many hours of robot video does that represent?

In [ ]:
# ── Exercise 1 Solution ──────────────────────────────────────────────────────
V = 4      # variations per task
E = 50     # episodes per variation
T = 200    # timesteps per episode
N_tasks = 100
fps = 30

episodes_per_task = V * E
timesteps_per_task = episodes_per_task * T
total_timesteps = N_tasks * timesteps_per_task
hours_of_video = total_timesteps / fps / 3600

print(f"(a) Episodes per task       = {V} × {E} = {episodes_per_task}")
print(f"(b) Timesteps per task      = {episodes_per_task} × {T} = {timesteps_per_task:,}")
print(f"(c) Total timesteps (100 tasks) = {total_timesteps:,}")
print(f"(d) At {fps} fps → {hours_of_video:.2f} hours of robot video")
print()
print("Key insight: Even 50 eps/variation generates enormous datasets — this is why\n"
      "'infinite demos' is genuinely useful for data-hungry deep learning.")

---

### Exercise 2 — Sparse Rewards and Exploration

With a **completely sparse reward** (+1 only on task completion), a random policy must stumble upon success by chance.

Assume the task workspace is a **10cm × 10cm × 10cm** cube. The robot must place an object within a **1cm × 1cm × 1cm** success zone.

a) If the robot places the object uniformly at random, what is the probability of success on a single attempt?  
b) On average, how many random attempts are needed before the first success?  
c) At 10 attempts per second, how many seconds on average before first success?  
d) This is why sparse rewards are hard — but what does RLBench provide to help?

In [ ]:
# ── Exercise 2 Solution ──────────────────────────────────────────────────────
V_workspace = 10**3   # cm^3
V_success   = 1**3    # cm^3

p_success = V_success / V_workspace
avg_attempts = 1 / p_success
attempts_per_sec = 10
avg_seconds = avg_attempts / attempts_per_sec

print(f"(a) P(success per attempt)  = {V_success}/{V_workspace} = {p_success:.4f} = {p_success*100:.2f}%")
print(f"(b) Avg attempts needed     = 1 / {p_success} = {avg_attempts:.0f} attempts")
print(f"(c) Avg time to first hit   = {avg_attempts:.0f} / {attempts_per_sec} = {avg_seconds:.1f} seconds")
print()
print("(d) To combat sparse rewards, RLBench provides:")
print("    - Expert demonstrations (π*) to bootstrap a good initial policy")
print("    - Motion-planned waypoints that guide the robot to the goal region")
print("    - Infinite demos so algorithms like DAGGER, BC+RL can pre-train before RL")

# Visualise the exploration problem
np.random.seed(7)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Exercise 2 — Sparse Rewards: Random vs Demo-Guided Exploration', fontsize=11)

# Random exploration — uniform
ax = axes[0]
n_pts = 2000
pts = np.random.uniform(0, 10, (n_pts, 2))
success_mask = (pts[:, 0] < 1) & (pts[:, 1] < 1)
ax.scatter(pts[~success_mask, 0], pts[~success_mask, 1], s=2, alpha=0.3, color='#90A4AE')
ax.scatter(pts[success_mask, 0],  pts[success_mask, 1],  s=20, color='#EF5350', label='Success!')
rect = plt.Rectangle((0,0), 1, 1, linewidth=2, edgecolor='red', facecolor='#FFCDD2', alpha=0.4)
ax.add_patch(rect)
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.set_title(f'A) Random Exploration (1% success zone)\n{success_mask.sum()} hits in {n_pts} attempts', fontsize=9)
ax.set_xlabel('x (cm)'); ax.set_ylabel('y (cm)')
ax.legend(fontsize=9)

# Demo-guided: Gaussian around goal
ax2 = axes[1]
pts2 = np.random.multivariate_normal([0.5, 0.5], [[0.8, 0], [0, 0.8]], n_pts)
pts2 = np.clip(pts2, 0, 10)
success2 = (pts2[:, 0] < 1) & (pts2[:, 1] < 1)
ax2.scatter(pts2[~success2, 0], pts2[~success2, 1], s=2, alpha=0.3, color='#90CAF9')
ax2.scatter(pts2[success2, 0],  pts2[success2, 1],  s=20, color='#4CAF50', label='Success!')
rect2 = plt.Rectangle((0,0), 1, 1, linewidth=2, edgecolor='red', facecolor='#FFCDD2', alpha=0.4)
ax2.add_patch(rect2)
ax2.set_xlim(0, 10); ax2.set_ylim(0, 10)
ax2.set_title(f'B) Demo-Guided Exploration (policy near goal)\n{success2.sum()} hits in {n_pts} attempts', fontsize=9)
ax2.set_xlabel('x (cm)'); ax2.set_ylabel('y (cm)')
ax2.legend(fontsize=9)

plt.tight_layout(); plt.show()
print(f"\nSpeedup from demos: {success2.sum() / max(success_mask.sum(),1):.1f}× more successes in same # attempts!")

---

### Exercise 3 — Few-Shot Generalisation Mathematics

In the few-shot challenge, a model is trained on **M = 90 tasks** and tested on **N = 10 unseen tasks**, each with **K demonstrations**.

a) What fraction of tasks are hidden at test time?  
b) If a model achieves **40% SR** at 5-shot and **60% SR** at 20-shot, what is the implied improvement per additional demo? (linear approximation)  
c) Plot the expected learning curve if SR improves **logarithmically** with K: $SR(K) = a \cdot \log(K) + b$, calibrated to 40% at K=5 and 60% at K=20.  
d) Using this model, what SR would we expect at K=100 and at K=1000?

In [ ]:
# ── Exercise 3 Solution ──────────────────────────────────────────────────────
M_train = 90; N_test = 10; total = 100
test_fraction = N_test / total

SR_5  = 40; SR_20 = 60  # percent
K_5   = 5;  K_20  = 20

# (b) Linear improvement
linear_slope = (SR_20 - SR_5) / (K_20 - K_5)

# (c) Log model: SR = a * log(K) + b
# Two equations: SR_5 = a*ln(5)+b, SR_20 = a*ln(20)+b
A = np.array([[np.log(K_5),  1],
              [np.log(K_20), 1]])
b_vec = np.array([SR_5, SR_20])
a_coef, b_coef = np.linalg.solve(A, b_vec)

# (d) Extrapolate
SR_100  = a_coef * np.log(100)  + b_coef
SR_1000 = a_coef * np.log(1000) + b_coef

print(f"(a) Test fraction = {N_test}/{total} = {test_fraction*100:.0f}%")
print(f"(b) Linear improvement = ({SR_20}-{SR_5}) / ({K_20}-{K_5}) = {linear_slope:.2f}% SR per demo")
print(f"(c) Log model: SR(K) = {a_coef:.3f}·ln(K) + {b_coef:.3f}")
print(f"    SR(5)  = {a_coef*np.log(5)+b_coef:.2f}% (check: should be {SR_5}%)")
print(f"    SR(20) = {a_coef*np.log(20)+b_coef:.2f}% (check: should be {SR_20}%)")
print(f"(d) SR(K=100)  ≈ {SR_100:.1f}%")
print(f"    SR(K=1000) ≈ {SR_1000:.1f}%")
print()
print("Insight: Logarithmic growth means performance improves quickly with first demos,")
print("then flattens — you can't just throw more demos at the problem indefinitely.")

# Plot
K_range = np.linspace(1, 200, 300)
SR_log  = a_coef * np.log(K_range) + b_coef
SR_lin  = linear_slope * (K_range - K_5) + SR_5

plt.figure(figsize=(9, 4))
plt.plot(K_range, SR_log, 'royalblue', linewidth=2.5, label=f'Log model: {a_coef:.2f}·ln(K)+{b_coef:.2f}')
plt.plot(K_range, SR_lin, 'darkorange', linestyle='--', linewidth=2, label='Linear model')
plt.scatter([5, 20], [SR_5, SR_20], s=100, c='red', zorder=5, label='Calibration points')
plt.axhline(100, color='gray', linestyle=':', lw=1.2, label='Perfect SR = 100%')
plt.xlabel('K (number of demonstrations)'); plt.ylabel('Success Rate (%)')
plt.title('Exercise 3 — K-Shot Learning Curve Models')
plt.legend(fontsize=9); plt.grid(alpha=0.3); plt.xlim(1, 200); plt.ylim(0, 110)
plt.tight_layout(); plt.show()

---

### Exercise 4 — Design Thinking (Open-Ended)

You want to add a new task to RLBench: **"Make a cup of tea"** — a realistic household robot task.

a) Break this task into sub-actions. Which would be separate **episodes** and which would be separate **variations**?  
b) Write pseudocode for the `init_task()`, `init_variation()`, and `init_episode()` functions.  
c) What **language descriptions** would you write for each variation?  
d) What makes this task hard for current RL/IL methods?

In [ ]:
# ── Exercise 4 — Guided Answer ────────────────────────────────────────────────

print("(a) Task breakdown for 'Make a cup of tea'")
print("-" * 60)

variations = {
    'Variation 0': {
        'description': 'Make black tea',
        'steps': ['Pick up kettle', 'Pour water into cup',
                  'Pick up teabag', 'Place teabag in cup', 'Wait', 'Remove teabag'],
        'language': ['make a cup of black tea', 'brew a black tea', 'prepare black tea']
    },
    'Variation 1': {
        'description': 'Make tea with milk',
        'steps': ['Pick up kettle', 'Pour water into cup',
                  'Pick up teabag', 'Place teabag in cup', 'Wait', 'Remove teabag',
                  'Pick up milk jug', 'Pour milk into cup'],
        'language': ['make a cup of tea with milk', 'brew tea with milk']
    },
    'Variation 2': {
        'description': 'Make tea with sugar',
        'steps': ['...all steps above...', 'Pick up sugar spoon', 'Add sugar', 'Stir'],
        'language': ['make sweet tea', 'brew tea with one sugar']
    }
}

for var, info in variations.items():
    print(f"  {var}: {info['description']}")
    print(f"    Steps:    {', '.join(info['steps'])}")
    print(f"    Language: {info['language']}")
    print()

print("Position-randomised PER EPISODE: kettle position, cup position, teabag box position")
print()

print("(b) Pseudocode:")
pseudocode = '''
def init_task(self):
    self.kettle  = Shape('kettle')
    self.cup     = Shape('cup')
    self.teabag  = Shape('teabag')
    self.register_graspable_objects([self.kettle, self.teabag])
    # Success: teabag inside cup, kettle returned to stand
    self.register_success_conditions([DetectedCondition(self.teabag, self.cup_sensor)])

def init_variation(self, index):
    if index == 0:
        return ['make a cup of black tea', 'brew a black tea']
    elif index == 1:
        self.milk_jug.set_visible(True)
        return ['make tea with milk']

def init_episode(self, index):
    # Randomise positions within workspace bounds
    self.kettle.set_position(random_pos(bounds=table_bounds))
    self.cup.set_position(random_pos(bounds=table_bounds))
'''
print(pseudocode)

print("(d) Why this is hard:")
difficulties = [
    'Long horizon: 6-9 sequential sub-tasks, each must succeed',
    'Liquid handling: pouring requires precise angle + velocity control',
    'Deformable object: teabag string dangles unpredictably',
    'Precision grasping: kettle handle has narrow grasp region',
    'Temporal dependency: must pour water BEFORE adding teabag',
    'Sparse reward: no signal until the teabag is correctly placed',
]
for d in difficulties:
    print(f'  • {d}')

---
## 🎉 Congratulations — You've Completed the RLBench Tutorial!

### What You've Learned

- **Why** a unified robot learning benchmark was needed in 2019 and what problems it solves
- **The 6 design properties** that guided every decision in RLBench
- **The 3-level hierarchy** of Task → Variation → Episode and the formal definitions
- **What the robot observes** (6 image streams + proprioception) and what actions it can take (7 choices)
- **How infinite demos** are generated via motion planning on waypoints
- **The Environment API** and the standard RL loop with sparse rewards
- **How to create new tasks** with just two files using PyRep
- **The few-shot challenge** setup (1-shot / 5-shot / 20-shot on 10 unseen tasks)
- **Five research directions** enabled by RLBench (RL, IL, sim-to-real, multi-task, SLAM)
- **How RLBench relates to AutoBio** — complementary benchmarks covering breadth vs depth

### Next Steps

1. 🔗 **Explore the benchmark:** https://sites.google.com/view/rlbench
2. 📄 **Read the full paper:** arXiv:1909.12271
3. 🤖 **Install and run RLBench:** `pip install rlbench` (requires V-REP/CoppeliaSim)
4. 🧪 **Try a task:** Use the API from Figure 5 to train a simple behavioural cloning agent
5. 💡 **Design a new task:** Use the task builder to add your own task and submit a PR!
6. 🔬 **Read follow-up work:** PerAct (2022), RVT (2023) — methods built directly on RLBench